# 🚗 Clasificación de Llantas Dañadas — Pregunta 1
## Redes Neuronales y Aprendizaje Profundo

Este notebook ejecuta el pipeline completo de entrenamiento y evaluación 
de modelos CNN para clasificar llantas en buen estado vs dañadas.

**Arquitecturas implementadas:**
- CustomCNN: diseñada desde cero con 4 bloques convolucionales
- ResNet-50: transfer learning con fine-tuning
- EfficientNet-B3: transfer learning con fine-tuning

**El notebook debe ejecutarse en orden, celda por celda.**

## Celda 1 — Verificar GPU disponible

Antes de cualquier entrenamiento verificamos que Colab nos asignó 
una GPU. Sin GPU el entrenamiento tomaría horas en lugar de minutos.

- `torch.cuda.is_available()` → confirma si hay GPU disponible
- `get_device_name(0)` → muestra el modelo de GPU (T4, A100, L4)
- `total_memory` → memoria total de la GPU en gigabytes

**Resultado esperado:** GPU disponible: True + nombre de la GPU

In [1]:
import torch

print("GPU disponible:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("Memoria:", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), "GB")

GPU disponible: True
GPU: Tesla T4
Memoria: 15.6 GB


## Celda 2 — Cargar el proyecto desde GitHub

El código de nuestro proyecto vive en GitHub. Lo clonamos directamente 
al servidor de Colab porque:

- El servidor de Colab es una máquina virtual en la nube de Google
- Esa máquina está vacía al inicio — no tiene nuestro código
- `git clone` descarga todos los archivos del repositorio a `/content/`
- `os.chdir` nos posiciona dentro de la carpeta del proyecto

**Estructura que se descarga:**
- `src/` → código fuente (modelos, dataset, entrenamiento)
- `configs/` → configuraciones YAML de cada experimento  
- `scripts/` → scripts de preparación y evaluación
- `reports/` → informe


**Resultado esperado:** ver las carpetas configs, src, scripts, reports

In [2]:
import os

# Clonar el repositorio desde GitHub
!git clone https://github.com/manadevelop/tire-classification.git \
    /content/tire_classification

# Posicionarse en el proyecto
os.chdir('/content/tire_classification')
print("✓ Proyecto cargado desde GitHub")
!ls

Cloning into '/content/tire_classification'...
remote: Enumerating objects: 34, done.
remote: Counting objects: 100% (34/34), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 34 (delta 1), reused 34 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (34/34), 37.88 KiB | 7.58 MiB/s, done.
Resolving deltas: 100% (1/1), done.
✓ Proyecto cargado desde GitHub
colab_training.ipynb  README.md  requirements.txt  scripts
configs		      reports	 run_all.sh	   src


## Celda 3 — Instalar dependencias del proyecto

Instalamos todas las librerías Python que necesita el proyecto.
Están listadas en `requirements.txt`. Las principales son:

| Librería | Versión | Para qué sirve |
|---|---|---|
| torch | ≥2.1.0 | Framework de deep learning — entrena las redes neuronales |
| torchvision | ≥0.16.0 | Modelos preentrenados (ResNet, EfficientNet) e imágenes |
| albumentations | ≥1.3.1 | Aumentación de datos avanzada orientada a texturas |
| opencv-python | ≥4.8.0 | Procesamiento de imágenes para Grad-CAM |
| scikit-learn | ≥1.3.0 | Métricas: F1, AUC-ROC, matriz de confusión |
| matplotlib | ≥3.7.0 | Gráficas de curvas de entrenamiento |
| seaborn | ≥0.12.0 | Visualización de matrices de confusión |
| tqdm | ≥4.65.0 | Barras de progreso durante el entrenamiento |
| pyyaml | ≥6.0 | Leer archivos de configuración YAML |

El flag `-q` significa "quiet" — instala sin mostrar mensajes innecesarios.

**Tiempo estimado:** 2-3 minutos

In [3]:
# Instalar todas las dependencias listadas en requirements.txt
# -q = modo silencioso (no muestra cada paso de la instalación)
!pip install -r requirements.txt -q

print("✓ Todas las dependencias instaladas correctamente")
print("\nVerificando versiones clave:")
import torch, torchvision, sklearn, albumentations
print(f"  PyTorch:       {torch.__version__}")
print(f"  Torchvision:   {torchvision.__version__}")
print(f"  Scikit-learn:  {sklearn.__version__}")
print(f"  Albumentations:{albumentations.__version__}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 113.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.5/12.5 MB 118.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 139.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 6.8 MB/s eta 0:00:00
✓ Todas las dependencias instaladas correctamente

Verificando versiones clave:
  PyTorch:       2.10.0+cu128
  Torchvision:   0.25.0+cu128
  Scikit-learn:  1.6.1
  Albumentations:2.0.8


El archivo kaggle.json descargado contiene:
- `username` → tu usuario de Kaggle
- `key` → tu API key personal (cadena larga de caracteres)

**Pasos que ejecuta esta celda:**
1. Crea el archivo `kaggle.json` directamente en `/root/.kaggle/` 
   usando tus credenciales pegadas en el código
2. `chmod 600` → permisos de seguridad requeridos por Kaggle CLI
3. `kaggle datasets download` → descarga el dataset completo
4. `--unzip` → descomprime automáticamente en `data/raw/`
5. Las imágenes quedan organizadas en `data/raw/good/` y `data/raw/cracked/`

**⚠️ Seguridad:** no subas este notebook a GitHub con tu API key 
visible. Antes de hacer git push, reemplaza tu key por `"TU_API_KEY"`.

**Tiempo estimado:** 3-5 minutos según velocidad de conexión

In [4]:
import os

# Pega aquí tu usuario y API key de Kaggle
KAGGLE_USERNAME = "mninadev"
KAGGLE_KEY      = "KGAT_bf5d574d96b8cc1e586f0c50c8f6493f"

# Crear el archivo kaggle.json directamente en Colab
os.makedirs('/root/.kaggle', exist_ok=True)

with open('/root/.kaggle/kaggle.json', 'w') as f:
    f.write(f'{{"username":"{KAGGLE_USERNAME}","key":"{KAGGLE_KEY}"}}')

!chmod 600 /root/.kaggle/kaggle.json

# Descargar dataset
print("⬇️ Descargando dataset...")
!kaggle datasets download \
    -d jehanbhathena/tire-texture-image-recognition \
    -p data/raw \
    --unzip

print("✓ Dataset descargado")
!ls data/raw/

⬇️ Descargando dataset...
Dataset URL: https://www.kaggle.com/datasets/jehanbhathena/tire-texture-image-recognition
License(s): CC0-1.0
100% 708M/708M [00:42<00:00, 17.4MB/s] 

✓ Dataset descargado
'Tire Textures'


## Celda 5 — Preparar y dividir el dataset

El dataset descargado está en `data/raw/` sin ninguna organización 
para entrenamiento. Necesitamos dividirlo en tres subconjuntos:

| Split | Porcentaje | Para qué sirve |
|---|---|---|
| train/ | 70% | El modelo **aprende** de estas imágenes |
| val/ | 15% | **Ajustamos** hiperparámetros durante el entrenamiento |
| test/ | 15% | **Medimos** el rendimiento final — imágenes nunca vistas |

**¿Por qué es importante esta división?**
Si evaluamos con las mismas imágenes que usamos para entrenar, 
el modelo parece mejor de lo que realmente es. El conjunto test 
simula el caso real: llantas que el modelo nunca vio antes.

**División estratificada:** garantiza que cada split mantenga 
la misma proporción de clases good/cracked, evitando sesgos.

---

### Estadísticas reales del dataset

| Clase | Total | Train (70%) | Val (15%) | Test (15%) |
|---|---|---|---|---|
| good | 491 | ~344 | ~74 | ~74 |
| cracked | 537 | ~376 | ~80 | ~80 |
| **Total** | **1,028** | **~720** | **~154** | **~154** |

**Ratio de desbalance: 1.09x**

Este valor indica que el dataset está **prácticamente balanceado** 
— solo 46 imágenes de diferencia entre clases. 

**Observación importante para el informe:**
A pesar del bajo desbalance en este dataset de laboratorio, 
aplicamos Focal Loss y WeightedRandomSampler como buenas 
prácticas, ya que en entornos industriales reales el desbalance 
suele ser mucho mayor (llantas buenas >> llantas dañadas).

**Tiempo estimado:** menos de 1 minuto

In [7]:
import os, shutil
from pathlib import Path

# ── Reorganizar dataset a la estructura esperada ──────────────
# Unimos training_data y testing_data en una sola carpeta por clase
# y renombramos "normal" → "good"

raw_base = Path("data/raw/Tire Textures")
dest_base = Path("data/raw_organized")

# Mapeo de nombres de clase
clase_map = {
    "normal":  "good",
    "cracked": "cracked"
}

# Crear carpetas destino
for clase in ["good", "cracked"]:
    (dest_base / clase).mkdir(parents=True, exist_ok=True)

# Copiar imágenes de training_data y testing_data
copiadas = {"good": 0, "cracked": 0}

for split in ["training_data", "testing_data"]:
    for clase_orig, clase_dest in clase_map.items():
        src = raw_base / split / clase_orig
        if not src.exists():
            continue
        for img in src.iterdir():
            if img.suffix.lower() in ['.jpg','.jpeg','.png','.bmp']:
                dst = dest_base / clase_dest / f"{split}_{img.name}"
                shutil.copy2(img, dst)
                copiadas[clase_dest] += 1

print("✓ Dataset reorganizado:")
print(f"  good:    {copiadas['good']} imágenes")
print(f"  cracked: {copiadas['cracked']} imágenes")
print(f"  Total:   {sum(copiadas.values())} imágenes")
ratio = max(copiadas.values()) / min(copiadas.values())
print(f"  Ratio de desbalance: {ratio:.2f}x")

# ── Ahora sí ejecutar prepare_data con la carpeta correcta ────
print("\n📊 Dividiendo en train/val/test (70/15/15)...")
!python scripts/prepare_data.py \
    --raw_dir data/raw_organized \
    --out_dir data/processed \
    --train_ratio 0.70 \
    --val_ratio 0.15

print("\n✓ Estructura final:")
!find data/processed -type d | sort

✓ Dataset reorganizado:
  good:    491 imágenes
  cracked: 537 imágenes
  Total:   1028 imágenes
  Ratio de desbalance: 1.09x

📊 Dividiendo en train/val/test (70/15/15)...
Leyendo imágenes desde: data/raw_organized
  [cracked] 537 imágenes encontradas
  [good] 491 imágenes encontradas

Dataset procesado en: data/processed
Ratio de desbalance: 1.09
Estadísticas guardadas en: data/processed/split_stats.json

✓ Estructura final:
data/processed
data/processed/test
data/processed/test/cracked
data/processed/test/good
data/processed/train
data/processed/train/cracked
data/processed/train/good
data/processed/val
data/processed/val/cracked
data/processed/val/good


## Celda 6 — Entrenar CustomCNN (desde cero)

Entrenamos la primera arquitectura: CNN diseñada completamente 
desde cero sin pesos preentrenados.

**Datos reales del dataset:**
- Total: 1,028 imágenes (good: 491 | cracked: 537)
- Desbalance: 1.09x (prácticamente balanceado)
- Train: 718 | Val: 154 | Test: 156

**Configuración:**
- Épocas máximas: 60
- Batch size: 32  
- Optimizador: AdamW (lr=3×10⁻⁴)
- Pérdida: Focal Loss (α=0.75, γ=2.0)
- Augmentación: aggressive
- Early stopping: 15 épocas sin mejora en F1

**Tiempo estimado con T4:** 15-20 minutos

In [8]:
print("="*60)
print("🧠 ENTRENANDO: CustomCNN (desde cero)")
print("="*60)
print("Configuración: configs/train_scratch.yaml\n")

!python src/train.py --config configs/train_scratch.yaml

print("\n✓ CustomCNN entrenada.")
print("Archivos guardados:")
!ls outputs/custom_cnn_aggressive_focal/

🧠 ENTRENANDO: CustomCNN (desde cero)
Configuración: configs/train_scratch.yaml

[2026-05-23 01:38:35][custom_cnn_aggressive_focal][INFO] Dispositivo: cuda
/content/tire_classification/src/data/transforms.py:96: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
/content/tire_classification/src/data/transforms.py:97: UserWarning: Argument(s) 'max_holes, max_height, max_width, min_holes, fill_value' are not valid for transform CoarseDropout
  A.CoarseDropout(
[2026-05-23 01:38:35][custom_cnn_aggressive_focal][INFO] Train: 718 | Val: 153 | Test: 157
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the 

In [9]:
import json

print("="*60)
print("RESULTADOS FINALES — CustomCNN")
print("="*60)

# Métricas en test
with open('outputs/custom_cnn_aggressive_focal/test_metrics.json') as f:
    test = json.load(f)

print(f"\n📊 MÉTRICAS EN TEST:")
print(f"  Accuracy : {test['accuracy']:.4f}")
print(f"  Precision: {test['precision']:.4f}")
print(f"  Recall   : {test['recall']:.4f}")
print(f"  F1-macro : {test['f1_macro']:.4f}")
print(f"  AUC-ROC  : {test['auc_roc']:.4f}")
print(f"\n📉 Matriz de confusión:")
print(f"  {test['confusion_matrix']}")
print(f"\n📋 Reporte completo:")
print(test['classification_report'])

# Mejor época
with open('outputs/custom_cnn_aggressive_focal/training_history.json') as f:
    hist = json.load(f)

best_f1 = max(hist['val_f1'])
best_epoch = hist['val_f1'].index(best_f1) + 1
total_epochs = len(hist['val_f1'])

print(f"\n📈 HISTORIAL:")
print(f"  Épocas entrenadas : {total_epochs}")
print(f"  Mejor F1 en val   : {best_f1:.4f} (época {best_epoch})")
print(f"  Último AUC        : {hist['val_auc'][-1]:.4f}")

RESULTADOS FINALES — CustomCNN

📊 MÉTRICAS EN TEST:
  Accuracy : 0.6306
  Precision: 0.6306
  Recall   : 0.6270
  F1-macro : 0.6262
  AUC-ROC  : 0.6891

📉 Matriz de confusión:
  [[58, 24], [34, 41]]

📋 Reporte completo:
              precision    recall  f1-score   support

           0       0.63      0.71      0.67        82
           1       0.63      0.55      0.59        75

    accuracy                           0.63       157
   macro avg       0.63      0.63      0.63       157
weighted avg       0.63      0.63      0.63       157


📈 HISTORIAL:
  Épocas entrenadas : 26
  Mejor F1 en val   : 0.6507 (época 11)
  Último AUC        : 0.7164


In [10]:
# Verificar versión de albumentations instalada
import albumentations as A
print("Versión albumentations:", A.__version__)

Versión albumentations: 2.0.8


In [11]:
# Fix para albumentations 2.0.8
# Reescribe transforms.py con los parámetros correctos

fix_code = '''
import torchvision.transforms as T
import albumentations as A
from albumentations.pytorch import ToTensorV2
import numpy as np
from PIL import Image
from typing import Callable

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

class AlbumentationsWrapper:
    def __init__(self, transform):
        self.transform = transform
    def __call__(self, img):
        if isinstance(img, Image.Image):
            img = np.array(img)
        result = self.transform(image=img)
        return result["image"]

def get_train_transforms(img_size=224, augmentation_strategy="standard"):
    if augmentation_strategy == "minimal":
        return T.Compose([
            T.Resize((img_size, img_size)),
            T.ToTensor(),
            T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])
    elif augmentation_strategy == "standard":
        return T.Compose([
            T.Resize((img_size + 32, img_size + 32)),
            T.RandomCrop(img_size),
            T.RandomHorizontalFlip(p=0.5),
            T.RandomVerticalFlip(p=0.3),
            T.RandomRotation(degrees=20),
            T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
            T.RandomGrayscale(p=0.05),
            T.ToTensor(),
            T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            T.RandomErasing(p=0.2, scale=(0.02, 0.1)),
        ])
    elif augmentation_strategy == "aggressive":
        pipeline = A.Compose([
            A.Resize(img_size + 32, img_size + 32),
            A.RandomCrop(img_size, img_size),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.3),
            A.Rotate(limit=25, p=0.6),
            A.ElasticTransform(p=0.3),
            A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.3),
            A.OpticalDistortion(p=0.2),
            A.RandomBrightnessContrast(
                brightness_limit=0.3, contrast_limit=0.3, p=0.6),
            A.HueSaturationValue(
                hue_shift_limit=10, sat_shift_limit=20, 
                val_shift_limit=20, p=0.4),
            A.RandomGamma(gamma_limit=(70, 130), p=0.4),
            A.CLAHE(clip_limit=4.0, p=0.3),
            # FIXED para albumentations 2.0.8
            A.GaussNoise(p=0.3),
            A.CoarseDropout(
                num_holes_range=(1, 8),
                hole_height_range=(8, 16),
                hole_width_range=(8, 16),
                p=0.3,
            ),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ])
        return AlbumentationsWrapper(pipeline)
    else:
        raise ValueError(f"Estrategia desconocida: {augmentation_strategy}")

def get_val_transforms(img_size=224):
    return T.Compose([
        T.Resize((img_size, img_size)),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])
'''

with open('src/data/transforms.py', 'w') as f:
    f.write(fix_code)

print("✓ transforms.py corregido para albumentations 2.0.8")

✓ transforms.py corregido para albumentations 2.0.8


In [12]:
# Corregir num_workers de 4 a 2 en todos los configs
import yaml

for config_file in ['configs/train_resnet50.yaml', 
                    'configs/train_efficientnet.yaml']:
    with open(config_file) as f:
        cfg = yaml.safe_load(f)
    
    cfg['training']['num_workers'] = 2
    
    with open(config_file, 'w') as f:
        yaml.dump(cfg, f, default_flow_style=False)
    
    print(f"✓ {config_file} → num_workers: 2")

✓ configs/train_resnet50.yaml → num_workers: 2
✓ configs/train_efficientnet.yaml → num_workers: 2


## Celda 7 — Entrenar ResNet-50 (Transfer Learning)

### ¿Qué es Transfer Learning?

En lugar de aprender desde cero como la CustomCNN, ResNet-50
ya fue entrenada previamente en **ImageNet** — una base de datos
de 1.2 millones de imágenes y 1,000 categorías diferentes.

Durante ese entrenamiento previo, ResNet-50 aprendió a detectar:

- **Capas iniciales:** bordes, líneas, esquinas simples
- **Capas intermedias:** texturas, patrones, formas complejas
- **Capas finales:** objetos completos y sus variaciones

Nosotros aprovechamos todo ese conocimiento y solo reemplazamos
la última capa para adaptarla a nuestras 2 clases: good/cracked.

---

### Arquitectura ResNet-50 modificada

El backbone de ResNet-50 procesa la imagen a través de 4 grupos
de bloques residuales con filtros crecientes:

1. Capa inicial: Conv(7×7, 64 filtros) + BN + ReLU + MaxPool
2. Layer 1: 3 bloques residuales → 256 filtros
3. Layer 2: 4 bloques residuales → 512 filtros
4. Layer 3: 6 bloques residuales → 1,024 filtros
5. Layer 4: 3 bloques residuales → 2,048 filtros
6. GlobalAvgPool → 2,048 features

Clasificador nuevo (entrenado desde cero):

1. Dropout(0.4)
2. Linear(2048 → 256) + ReLU
3. Dropout(0.2)
4. Linear(256 → 2) → salida: P(good) y P(cracked)

---

### ¿Qué es un bloque residual?

Es la innovación clave de ResNet. En lugar de aprender la
transformación completa de la entrada, aprende solo la
**diferencia** (residuo) respecto a la entrada original:

- **Red normal:** salida = F(entrada)
- **Red residual:** salida = F(entrada) + entrada

La suma con la entrada original se llama **skip connection**
o conexión de atajo. Esto permite entrenar redes muy profundas
sin que los gradientes desaparezcan durante el backpropagation,
problema conocido como **vanishing gradient**.

---

### Comparación con CustomCNN

| Aspecto | CustomCNN | ResNet-50 |
|---|---|---|
| Parámetros | 1.8M | 24.0M |
| Pesos iniciales | Aleatorios | **ImageNet** |
| Conocimiento previo | Ninguno | **1.2M imágenes** |
| LR de entrenamiento | 3×10⁻⁴ | **1×10⁻⁴** (más baja) |
| Épocas máximas | 60 | 40 |

La tasa de aprendizaje es más baja en ResNet-50 para no
**destruir** el conocimiento preaprendido de ImageNet.

---

### Configuración (train_resnet50.yaml)

- **Épocas máximas:** 40
- **Batch size:** 32
- **Optimizador:** AdamW (lr = 1×10⁻⁴)
- **Pérdida:** Focal Loss (α=0.75, γ=2.0)
- **Augmentación:** aggressive
- **num_workers:** 2
- **Early stopping:** 12 épocas sin mejora en F1

---

### Resultados obtenidos

- **Accuracy :** 0.9682
- **Precision:** 0.9684
- **Recall   :** 0.9678
- **F1-macro :** 0.9681
- **AUC-ROC  :** 0.9894
- **Épocas entrenadas:** 30 (mejor modelo en época 18)
- **Tiempo de entrenamiento:** 16.5 minutos en GPU T4

### Mejora sobre CustomCNN

- F1-macro : +34.2 puntos porcentuales
- AUC-ROC  : +30.0 puntos porcentuales

Esta mejora dramática confirma que el transfer learning es
**indispensable** cuando el dataset es pequeño (1,028 imágenes).
ResNet-50 ya conocía patrones de textura desde ImageNet y solo
necesitó ajustar el clasificador final a nuestras 2 clases.

### ¿Qué es un bloque residual?

Es la innovación clave de ResNet. En lugar de aprender 
la transformación directa, aprende la **diferencia** (residuo):

Esto permite entrenar redes muy profundas sin que los gradientes 
desaparezcan durante el backpropagation.

---

### Comparación con CustomCNN

| Aspecto | CustomCNN | ResNet-50 |
|---|---|---|
| Parámetros | 1.8M | 25.6M |
| Pesos iniciales | Aleatorios | ImageNet |
| Conocimiento previo | Ninguno | 1.2M imágenes |
| LR de entrenamiento | 3×10⁻⁴ | 1×10⁻⁴ (más baja) |
| Épocas máximas | 60 | 40 |

La tasa de aprendizaje es más baja en ResNet-50 para no 
**destruir** el conocimiento preaprendido de ImageNet.

---

### Configuración (train_resnet50.yaml)
- **Épocas máximas:** 40
- **Batch size:** 32
- **Optimizador:** AdamW (lr=1×10⁻⁴)
- **Pérdida:** Focal Loss (α=0.75, γ=2.0)
- **Augmentación:** aggressive (ya corregida para albumentations 2.0.8)
- **num_workers:** 2 (corregido)
- **Early stopping:** 12 épocas sin mejora en F1

### Resultado esperado
ResNet-50 debería superar significativamente a CustomCNN 
gracias al conocimiento previo de ImageNet.
Esperamos F1 > 0.85 y AUC > 0.90

**Tiempo estimado con T4:** 20-25 minutos

In [13]:
print("="*60)
print("🔄 ENTRENANDO: ResNet-50 (Transfer Learning)")
print("="*60)
print("Configuración : configs/train_resnet50.yaml")
print("Backbone      : ResNet-50 preentrenado en ImageNet-1K V2")
print("Parámetros    : 25.6M")
print("LR            : 1×10⁻⁴ (baja para preservar pesos ImageNet)\n")

!python src/train.py --config configs/train_resnet50.yaml

print("\n✓ ResNet-50 entrenado.")
print("\nArchivos guardados:")
!ls outputs/resnet50_finetune_focal/

# Mostrar métricas finales automáticamente
import json

print("\n" + "="*60)
print("📊 RESULTADOS FINALES — ResNet-50")
print("="*60)

with open('outputs/resnet50_finetune_focal/test_metrics.json') as f:
    test = json.load(f)

print(f"\n  Accuracy : {test['accuracy']:.4f}")
print(f"  Precision: {test['precision']:.4f}")
print(f"  Recall   : {test['recall']:.4f}")
print(f"  F1-macro : {test['f1_macro']:.4f}")
print(f"  AUC-ROC  : {test['auc_roc']:.4f}")

print(f"\n📉 Matriz de confusión:")
cm = test['confusion_matrix']
print(f"  Verdaderos good (TN)   : {cm[0][0]}")
print(f"  Good → Cracked (FP)    : {cm[0][1]}")
print(f"  Cracked → Good (FN)    : {cm[1][0]}  ← más peligroso")
print(f"  Verdaderos cracked (TP): {cm[1][1]}")

with open('outputs/resnet50_finetune_focal/training_history.json') as f:
    hist = json.load(f)

best_f1    = max(hist['val_f1'])
best_epoch = hist['val_f1'].index(best_f1) + 1
total_ep   = len(hist['val_f1'])

print(f"\n📈 HISTORIAL:")
print(f"  Épocas entrenadas : {total_ep}")
print(f"  Mejor F1 en val   : {best_f1:.4f} (época {best_epoch})")

# Comparación con CustomCNN
print(f"\n{'='*60}")
print(f"⚖️  COMPARACIÓN: CustomCNN vs ResNet-50")
print(f"{'='*60}")
print(f"{'Métrica':<12} {'CustomCNN':>12} {'ResNet-50':>12} {'Mejora':>10}")
print(f"{'-'*48}")

custom_metrics = {
    'Accuracy' : 0.6306,
    'Precision': 0.6306,
    'Recall'   : 0.6270,
    'F1-macro' : 0.6262,
    'AUC-ROC'  : 0.6891,
}
resnet_metrics = {
    'Accuracy' : test['accuracy'],
    'Precision': test['precision'],
    'Recall'   : test['recall'],
    'F1-macro' : test['f1_macro'],
    'AUC-ROC'  : test['auc_roc'],
}

for metrica in custom_metrics:
    c = custom_metrics[metrica]
    r = resnet_metrics[metrica]
    mejora = (r - c) * 100
    signo = "+" if mejora >= 0 else ""
    print(f"{metrica:<12} {c:>12.4f} {r:>12.4f} {signo}{mejora:>8.1f}pp")

🔄 ENTRENANDO: ResNet-50 (Transfer Learning)
Configuración : configs/train_resnet50.yaml
Backbone      : ResNet-50 preentrenado en ImageNet-1K V2
Parámetros    : 25.6M
LR            : 1×10⁻⁴ (baja para preservar pesos ImageNet)

[2026-05-23 02:09:35][resnet50_finetune_focal][INFO] Dispositivo: cuda
[2026-05-23 02:09:35][resnet50_finetune_focal][INFO] Train: 718 | Val: 153 | Test: 157
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100% 97.8M/97.8M [00:00<00:00, 146MB/s] 
[2026-05-23 02:10:08][resnet50_finetune_focal][INFO] Parámetros totales: 24,033,090
[2026-05-23 02:10:08][resnet50_finetune_focal][INFO] Iniciando entrenamiento por 40 épocas...
[2026-05-23 02:10:40][resnet50_finetune_focal][INFO] Epoch 001/40 | TrainLoss=0.1410 | ValLoss=0.0694 | Acc=0.8562 | F1=0.8557 | AUC=0.9175
[2026-05-23 02:10:40][resnet50_finetune_focal][INFO]   ✓ Nuevo mejor modelo (F1=0.8557)
[2026-05-23 02:11:14][resnet50_fin

## Celda 8 — Entrenar EfficientNet-B3 (Transfer Learning)

### ¿Qué hace especial a EfficientNet-B3?

Mientras ResNet-50 simplemente apila más capas para mejorar,
EfficientNet usa una estrategia más inteligente llamada
**escalado compuesto**: escala simultáneamente tres dimensiones:

- **Profundidad** → más capas en la red
- **Anchura** → más filtros por capa
- **Resolución** → imágenes de entrada más grandes (300×300)

Las tres dimensiones se escalan juntas de forma balanceada,
logrando mejor rendimiento con menos parámetros.

---

### Comparación de arquitecturas

| Aspecto | CustomCNN | ResNet-50 | EfficientNet-B3 |
|---|---|---|---|
| Parámetros | 1.8M | 24.0M | **12.2M** |
| Resolución entrada | 224×224 | 224×224 | **300×300** |
| Pesos iniciales | Aleatorios | ImageNet | ImageNet |
| Top-1 en ImageNet | — | 76.1% | **82.1%** |

EfficientNet-B3 logra mayor precisión en ImageNet con
**la mitad de parámetros** que ResNet-50, y usa resolución
300×300 que captura más detalle fino de textura de llanta.

---

### Arquitectura EfficientNet-B3 modificada

El backbone original procesa la imagen a través de bloques
MBConv con atención SE (Squeeze-and-Excitation), extrayendo
1,536 features al final. Reemplazamos el clasificador por:

1. Dropout(0.35)
2. Linear(1536 → 256) + ReLU
3. Dropout(0.175)
4. Linear(256 → 2) → salida: P(good) y P(cracked)

### ¿Qué es MBConv con SE Attention?

Es el bloque fundamental de EfficientNet con dos partes:

**MBConv (Mobile Inverted Bottleneck):**
Expande los canales, aplica convolución depthwise y los comprime.
Mucho más eficiente que las convoluciones estándar de ResNet.

**SE (Squeeze and Excitation):**
Aprende qué canales son más importantes para cada imagen.
Recalibra las activaciones dando más peso a los canales útiles.
Especialmente bueno para texturas — muy relevante para llantas.

---

### Configuración (train_efficientnet.yaml)

- **Épocas máximas:** 40
- **Batch size:** 24 (menor por la resolución 300×300)
- **Optimizador:** AdamW (lr = 8×10⁻⁵)
- **Pérdida:** Focal Loss (α=0.75, γ=2.0)
- **Augmentación:** aggressive
- **Early stopping:** 12 épocas sin mejora en F1

### Resultado esperado

EfficientNet-B3 debería competir muy cerca de ResNet-50
gracias a su mayor resolución de entrada (300×300) que
captura más detalle de textura superficial de la llanta.
Con menos parámetros pero mejor arquitectura esperamos
**F1 ≥ 0.97** y **AUC ≥ 0.99**

**Tiempo estimado con T4:** 25-30 minutos

In [14]:
print("="*60)
print("⚡ ENTRENANDO: EfficientNet-B3 (Transfer Learning)")
print("="*60)
print("Configuración : configs/train_efficientnet.yaml")
print("Backbone      : EfficientNet-B3 preentrenado en ImageNet")
print("Parámetros    : 12.2M (mitad que ResNet-50)")
print("Resolución    : 300×300 px (mayor detalle de textura)\n")

!python src/train.py --config configs/train_efficientnet.yaml

print("\n✓ EfficientNet-B3 entrenado.")
print("\nArchivos guardados:")
!ls outputs/efficientnet_b3_finetune_focal/

# ── Métricas finales ───────────────────────────────────────────
import json

print("\n" + "="*60)
print("📊 RESULTADOS FINALES — EfficientNet-B3")
print("="*60)

with open('outputs/efficientnet_b3_finetune_focal/test_metrics.json') as f:
    test_eff = json.load(f)

print(f"\n  Accuracy : {test_eff['accuracy']:.4f}")
print(f"  Precision: {test_eff['precision']:.4f}")
print(f"  Recall   : {test_eff['recall']:.4f}")
print(f"  F1-macro : {test_eff['f1_macro']:.4f}")
print(f"  AUC-ROC  : {test_eff['auc_roc']:.4f}")

print(f"\n📉 Matriz de confusión:")
cm = test_eff['confusion_matrix']
print(f"  Verdaderos good (TN)   : {cm[0][0]}")
print(f"  Good → Cracked (FP)    : {cm[0][1]}")
print(f"  Cracked → Good (FN)    : {cm[1][0]}  ← más peligroso")
print(f"  Verdaderos cracked (TP): {cm[1][1]}")

with open('outputs/efficientnet_b3_finetune_focal/training_history.json') as f:
    hist_eff = json.load(f)

best_f1    = max(hist_eff['val_f1'])
best_epoch = hist_eff['val_f1'].index(best_f1) + 1
total_ep   = len(hist_eff['val_f1'])

print(f"\n📈 HISTORIAL:")
print(f"  Épocas entrenadas : {total_ep}")
print(f"  Mejor F1 en val   : {best_f1:.4f} (época {best_epoch})")

# ── Tabla comparativa final 3 modelos ─────────────────────────
print(f"\n{'='*60}")
print(f"🏆 COMPARACIÓN FINAL — 3 MODELOS")
print(f"{'='*60}")
print(f"{'Métrica':<12} {'CustomCNN':>11} {'ResNet-50':>11} {'EffNet-B3':>11}")
print(f"{'-'*47}")

metricas = ['accuracy', 'precision', 'recall', 'f1_macro', 'auc_roc']
nombres  = ['Accuracy', 'Precision', 'Recall', 'F1-macro', 'AUC-ROC']

custom = {
    'accuracy' : 0.6306,
    'precision': 0.6306,
    'recall'   : 0.6270,
    'f1_macro' : 0.6262,
    'auc_roc'  : 0.6891,
}

with open('outputs/resnet50_finetune_focal/test_metrics.json') as f:
    test_res = json.load(f)

for nombre, metrica in zip(nombres, metricas):
    c = custom[metrica]
    r = test_res[metrica]
    e = test_eff[metrica]

    mejor = max(c, r, e)
    c_str = f"{'★' if c == mejor else ' '}{c:.4f}"
    r_str = f"{'★' if r == mejor else ' '}{r:.4f}"
    e_str = f"{'★' if e == mejor else ' '}{e:.4f}"

    print(f"{nombre:<12} {c_str:>11} {r_str:>11} {e_str:>11}")

print(f"\n  ★ = mejor resultado en esa métrica")

# ── Veredicto final ────────────────────────────────────────────
candidatos = [
    ('CustomCNN',     custom['f1_macro']),
    ('ResNet-50',     test_res['f1_macro']),
    ('EfficientNet-B3', test_eff['f1_macro']),
]
ganador = max(candidatos, key=lambda x: x[1])

print(f"\n{'='*60}")
print(f"🏆 MEJOR MODELO: {ganador[0]}")
print(f"   F1-macro en test : {ganador[1]:.4f}")
print(f"   Recomendado para producción")
print(f"{'='*60}")

⚡ ENTRENANDO: EfficientNet-B3 (Transfer Learning)
Configuración : configs/train_efficientnet.yaml
Backbone      : EfficientNet-B3 preentrenado en ImageNet
Parámetros    : 12.2M (mitad que ResNet-50)
Resolución    : 300×300 px (mayor detalle de textura)

[2026-05-23 02:41:37][efficientnet_b3_finetune_focal][INFO] Dispositivo: cuda
[2026-05-23 02:41:37][efficientnet_b3_finetune_focal][INFO] Train: 718 | Val: 153 | Test: 157
Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth
100% 47.2M/47.2M [00:00<00:00, 191MB/s]
[2026-05-23 02:42:13][efficientnet_b3_finetune_focal][INFO] Parámetros totales: 11,090,218
[2026-05-23 02:42:13][efficientnet_b3_finetune_focal][INFO] Iniciando entrenamiento por 40 épocas...
[2026-05-23 02:42:51][efficientnet_b3_finetune_focal][INFO] Epoch 001/40 | TrainLoss=0.1375 | ValLoss=0.0869 | Acc=0.7778 | F1=0.7778 | AUC=0.8557
[2026-05-23 02:42:52][effici

In [15]:
import json

# ── EfficientNet-B3 métricas completas ────────────────────────
with open('outputs/efficientnet_b3_finetune_focal/test_metrics.json') as f:
    test_eff = json.load(f)

with open('outputs/efficientnet_b3_finetune_focal/training_history.json') as f:
    hist_eff = json.load(f)

with open('outputs/resnet50_finetune_focal/test_metrics.json') as f:
    test_res = json.load(f)

best_f1    = max(hist_eff['val_f1'])
best_epoch = hist_eff['val_f1'].index(best_f1) + 1
total_ep   = len(hist_eff['val_f1'])

print("="*60)
print("📊 RESULTADOS FINALES — EfficientNet-B3")
print("="*60)
print(f"\n  Accuracy : {test_eff['accuracy']:.4f}")
print(f"  Precision: {test_eff['precision']:.4f}")
print(f"  Recall   : {test_eff['recall']:.4f}")
print(f"  F1-macro : {test_eff['f1_macro']:.4f}")
print(f"  AUC-ROC  : {test_eff['auc_roc']:.4f}")

cm = test_eff['confusion_matrix']
print(f"\n📉 Matriz de confusión:")
print(f"  Verdaderos good (TN)   : {cm[0][0]}")
print(f"  Good → Cracked (FP)    : {cm[0][1]}")
print(f"  Cracked → Good (FN)    : {cm[1][0]}  ← más peligroso")
print(f"  Verdaderos cracked (TP): {cm[1][1]}")

print(f"\n📈 HISTORIAL:")
print(f"  Épocas entrenadas : {total_ep}")
print(f"  Mejor F1 en val   : {best_f1:.4f} (época {best_epoch})")
print(f"\n{test_eff['classification_report']}")

# ── Tabla comparativa final ────────────────────────────────────
print("="*60)
print("🏆 COMPARACIÓN FINAL — 3 MODELOS")
print("="*60)
print(f"{'Métrica':<12} {'CustomCNN':>11} {'ResNet-50':>11} {'EffNet-B3':>11}")
print(f"{'-'*47}")

custom = {
    'accuracy' : 0.6306,
    'precision': 0.6306,
    'recall'   : 0.6270,
    'f1_macro' : 0.6262,
    'auc_roc'  : 0.6891,
}
metricas = ['accuracy','precision','recall','f1_macro','auc_roc']
nombres  = ['Accuracy','Precision','Recall','F1-macro','AUC-ROC']

for nombre, metrica in zip(nombres, metricas):
    c = custom[metrica]
    r = test_res[metrica]
    e = test_eff[metrica]
    mejor = max(c, r, e)
    c_str = f"{'★' if c == mejor else ' '}{c:.4f}"
    r_str = f"{'★' if r == mejor else ' '}{r:.4f}"
    e_str = f"{'★' if e == mejor else ' '}{e:.4f}"
    print(f"{nombre:<12} {c_str:>11} {r_str:>11} {e_str:>11}")

print(f"\n  ★ = mejor resultado en esa métrica")

# ── Parámetros y eficiencia ────────────────────────────────────
print(f"\n{'='*60}")
print(f"⚙️  EFICIENCIA DE MODELOS")
print(f"{'='*60}")
print(f"{'Modelo':<16} {'Params':>8} {'F1':>8} {'AUC':>8} {'Épocas':>8}")
print(f"{'-'*52}")
print(f"{'CustomCNN':<16} {'1.8M':>8} {0.6262:>8.4f} {0.6891:>8.4f} {'26':>8}")
print(f"{'ResNet-50':<16} {'24.0M':>8} {test_res['f1_macro']:>8.4f} {test_res['auc_roc']:>8.4f} {'30':>8}")
print(f"{'EfficientNet-B3':<16} {'11.1M':>8} {test_eff['f1_macro']:>8.4f} {test_eff['auc_roc']:>8.4f} {total_ep:>8}")

# ── Mejoras sobre CustomCNN ────────────────────────────────────
print(f"\n{'='*60}")
print(f"📈 MEJORA SOBRE CustomCNN (línea base)")
print(f"{'='*60}")
for modelo, f1, auc in [
    ('ResNet-50',     test_res['f1_macro'], test_res['auc_roc']),
    ('EfficientNet',  test_eff['f1_macro'], test_eff['auc_roc']),
]:
    df1 = (f1  - custom['f1_macro']) * 100
    dau = (auc - custom['auc_roc'])  * 100
    print(f"  {modelo:<14} F1: +{df1:.1f}pp  |  AUC: +{dau:.1f}pp")

📊 RESULTADOS FINALES — EfficientNet-B3

  Accuracy : 0.9682
  Precision: 0.9680
  Recall   : 0.9689
  F1-macro : 0.9681
  AUC-ROC  : 0.9878

📉 Matriz de confusión:
  Verdaderos good (TN)   : 78
  Good → Cracked (FP)    : 4
  Cracked → Good (FN)    : 1  ← más peligroso
  Verdaderos cracked (TP): 74

📈 HISTORIAL:
  Épocas entrenadas : 40
  Mejor F1 en val   : 0.9673 (época 32)

              precision    recall  f1-score   support

           0       0.99      0.95      0.97        82
           1       0.95      0.99      0.97        75

    accuracy                           0.97       157
   macro avg       0.97      0.97      0.97       157
weighted avg       0.97      0.97      0.97       157

🏆 COMPARACIÓN FINAL — 3 MODELOS
Métrica        CustomCNN   ResNet-50   EffNet-B3
-----------------------------------------------
Accuracy          0.6306     ★0.9682     ★0.9682
Precision         0.6306     ★0.9684      0.9680
Recall            0.6270      0.9678     ★0.9689
F1-macro          

## Celda 9 — Evaluación y Visualización Grad-CAM

### ¿Qué es Grad-CAM?

Grad-CAM (Gradient-weighted Class Activation Mapping) es una
técnica de interpretabilidad que responde a la pregunta:

**¿Qué parte de la imagen miró el modelo para tomar su decisión?**

Funciona en 3 pasos:

1. **Forward pass:** pasa la imagen por la red y obtiene la predicción
2. **Backward pass:** calcula los gradientes de la clase predicha
   respecto a los mapas de activación de la última capa convolucional
3. **Mapa de calor:** pondera los mapas de activación por sus
   gradientes y aplica ReLU — las zonas más rojas son las más
   importantes para la decisión del modelo

### Fórmula matemática

El mapa Grad-CAM para la clase c se calcula como:

- Pesos de importancia: α = promedio global de los gradientes
- Mapa de calor: L = ReLU( suma ponderada de mapas de activación )
- Normalización a [0, 1] e interpolación al tamaño original

### ¿Por qué es importante para el informe?

Grad-CAM nos permite validar que el modelo aprendió correctamente:

- Para **cracked:** debe activarse sobre las grietas y zonas
  de desgaste irregular — no sobre el fondo o los bordes
- Para **good:** las activaciones deben distribuirse sobre el
  patrón regular de surcos de la llanta

Si el modelo activa zonas incorrectas, significaría que aprendió
un sesgo espurio en lugar de la textura real de daño.

---

### ¿Qué genera esta celda?

Para cada modelo entrenado se generan:

- **Grad-CAM correcto good:** 4 ejemplos bien clasificados
- **Grad-CAM correcto cracked:** 4 ejemplos bien clasificados
- **Análisis de 5 fallos:** los peores errores con hipótesis
- **Curva ROC comparativa:** los 3 modelos en un solo gráfico
- **Matriz de confusión:** visualización comentada por modelo

Todos los archivos se guardan en la carpeta `results/`
para incluirlos directamente en el informe.

**Tiempo estimado:** 5-10 minutos

In [18]:
# ── Fix gradcam.py + Regenerar Grad-CAM ───────────────────────

# PASO 1: Corregir gradcam.py
fix_gradcam = '''from pathlib import Path
from typing import List, Optional, Tuple
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
import cv2
from data.transforms import get_val_transforms, IMAGENET_MEAN, IMAGENET_STD


def overlay_heatmap(img_np, cam, alpha=0.45, colormap=cv2.COLORMAP_JET):
    heatmap = np.uint8(255 * cam)
    heatmap = cv2.applyColorMap(heatmap, colormap)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    if heatmap.shape[:2] != img_np.shape[:2]:
        heatmap = cv2.resize(heatmap, (img_np.shape[1], img_np.shape[0]))
    return (alpha * heatmap + (1 - alpha) * img_np).astype(np.uint8)


class GradCAMVisualizer:
    def __init__(self, model, device="cpu", img_size=224, class_names=None):
        self.model       = model.to(device)
        self.device      = device
        self.img_size    = img_size
        self.transform   = get_val_transforms(img_size)
        self.class_names = class_names or ["good", "cracked"]

    def _prepare_tensor(self, img_path):
        img_pil = Image.open(img_path).convert("RGB")
        img_np  = np.array(img_pil.resize((self.img_size, self.img_size)))
        tensor  = self.transform(img_pil).unsqueeze(0).to(self.device)
        return tensor, img_np

    def predict_and_explain(self, img_path, class_idx=None):
        tensor, img_np = self._prepare_tensor(img_path)
        cam = self.model.get_gradcam_map(tensor, class_idx)
        if isinstance(cam, torch.Tensor):
            cam = cam.detach().cpu().numpy()
        with torch.no_grad():
            logits = self.model(tensor)
            proba  = torch.softmax(logits, dim=1).squeeze().cpu()
        pred_cls  = proba.argmax().item()
        pred_conf = proba[pred_cls].item()
        return pred_cls, pred_conf, cam, img_np

    def visualize_grid(self, img_paths, true_labels=None,
                       save_path=None, n_cols=4):
        n      = len(img_paths)
        n_rows = (n + n_cols - 1) // n_cols
        fig, axes = plt.subplots(
            n_rows, n_cols * 2,
            figsize=(n_cols * 6, n_rows * 4)
        )
        axes = axes.flatten()
        for i, img_path in enumerate(img_paths):
            pred_cls, conf, cam, img_np = self.predict_and_explain(img_path)
            overlay = overlay_heatmap(img_np, cam)
            ax_img  = axes[i * 2]
            ax_cam  = axes[i * 2 + 1]
            ax_img.imshow(img_np); ax_img.axis("off")
            ax_img.set_title(
                f"Real: {self.class_names[true_labels[i]]}"
                if true_labels else "", fontsize=8)
            ax_cam.imshow(overlay); ax_cam.axis("off")
            if true_labels:
                ok = "✓" if pred_cls == true_labels[i] else "✗"
                ax_cam.set_title(
                    f"{ok} Pred: {self.class_names[pred_cls]} ({conf:.1%})",
                    fontsize=8)
        for j in range(i * 2 + 2, len(axes)):
            axes[j].axis("off")
        plt.suptitle("Grad-CAM: Activaciones del modelo",
                     fontsize=13, fontweight="bold")
        plt.tight_layout()
        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches="tight")
            plt.close()

    def analyze_failures(self, img_paths, true_labels,
                         save_dir, n_examples=5):
        save_dir = Path(save_dir)
        save_dir.mkdir(parents=True, exist_ok=True)
        failures = []
        for path, true_lbl in zip(img_paths, true_labels):
            pred_cls, conf, cam, img_np = self.predict_and_explain(path)
            if pred_cls != true_lbl:
                failures.append(
                    (conf, path, true_lbl, pred_cls, cam, img_np))
        failures.sort(key=lambda x: -x[0])
        failures = failures[:n_examples]
        print(f"  Fallos encontrados: {len(failures)}")
        for rank, (conf, path, true_lbl, pred_cls, cam, img_np) \
                in enumerate(failures, 1):
            overlay = overlay_heatmap(img_np, cam)
            fig, axes = plt.subplots(1, 3, figsize=(12, 4))
            axes[0].imshow(img_np);    axes[0].axis("off")
            axes[0].set_title(f"Real: {self.class_names[true_lbl]}")
            axes[1].imshow(cam, cmap="jet"); axes[1].axis("off")
            axes[1].set_title("Grad-CAM")
            axes[2].imshow(overlay);   axes[2].axis("off")
            tipo = "Positivo" if pred_cls == 1 else "Negativo"
            axes[2].set_title(
                f"Pred: {self.class_names[pred_cls]} ({conf:.1%})\\n"
                f"[Falso {tipo}]")
            fig.suptitle(
                f"Fallo #{rank}: {Path(path).name}", fontsize=10)
            plt.tight_layout()
            plt.savefig(
                save_dir / f"failure_{rank:02d}_{Path(path).stem}.png",
                dpi=150, bbox_inches="tight")
            plt.close()
            print(f"    [{rank}] Real:{self.class_names[true_lbl]} "
                  f"→ Pred:{self.class_names[pred_cls]} "
                  f"(conf={conf:.1%})")
        return failures
'''

with open('src/visualization/gradcam.py', 'w') as f:
    f.write(fix_gradcam)
print("✓ gradcam.py corregido")

# PASO 2: Recargar módulo corregido
import sys
for mod in list(sys.modules.keys()):
    if 'gradcam' in mod:
        del sys.modules[mod]

from visualization.gradcam import GradCAMVisualizer
print("✓ Módulo recargado")

# PASO 3: Generar Grad-CAM para los 3 modelos
print("\n🔥 Generando mapas Grad-CAM...")

modelos_cfg = [
    {'nombre':'CustomCNN',      'tipo':'custom',         'img_size':224},
    {'nombre':'ResNet-50',      'tipo':'resnet50',       'img_size':224},
    {'nombre':'EfficientNet-B3','tipo':'efficientnet_b3','img_size':300},
]

for cfg in modelos_cfg:
    nombre  = cfg['nombre']
    modelo  = modelos[nombre]
    out_dir = (f"results/"
               f"{nombre.lower().replace('-','').replace(' ','_')}")
    os.makedirs(out_dir, exist_ok=True)

    viz = GradCAMVisualizer(
        model=modelo,
        device=device,
        img_size=cfg['img_size'],
        class_names=['good', 'cracked'],
    )

    res   = resultados[nombre]
    paths = res['paths']
    preds = res['y_pred']
    true  = res['y_true']

    # Grad-CAM clase good
    good_ok = [p for p, pr, t in zip(paths, preds, true)
               if t == 0 and pr == 0][:4]
    if good_ok:
        viz.visualize_grid(
            img_paths=good_ok,
            true_labels=[0] * len(good_ok),
            save_path=f"{out_dir}/gradcam_good.png",
        )
        print(f"  ✓ {nombre}: gradcam_good.png")

    # Grad-CAM clase cracked
    crack_ok = [p for p, pr, t in zip(paths, preds, true)
                if t == 1 and pr == 1][:4]
    if crack_ok:
        viz.visualize_grid(
            img_paths=crack_ok,
            true_labels=[1] * len(crack_ok),
            save_path=f"{out_dir}/gradcam_cracked.png",
        )
        print(f"  ✓ {nombre}: gradcam_cracked.png")

    # Análisis de 5 fallos
    viz.analyze_failures(
        img_paths=list(paths),
        true_labels=list(true),
        save_dir=f"{out_dir}/failures",
        n_examples=5,
    )
    print(f"  ✓ {nombre}: failures guardados")

print("\n✅ Grad-CAM completo")
print("   Continúa con Celda 10 para descargar todo a tu Mac")

✓ gradcam.py corregido
✓ Módulo recargado

🔥 Generando mapas Grad-CAM...
  ✓ CustomCNN: gradcam_good.png
  ✓ CustomCNN: gradcam_cracked.png
  Fallos encontrados: 5
    [1] Real:cracked → Pred:good (conf=54.0%)
    [2] Real:cracked → Pred:good (conf=53.3%)
    [3] Real:cracked → Pred:good (conf=52.9%)
    [4] Real:good → Pred:cracked (conf=52.8%)
    [5] Real:good → Pred:cracked (conf=52.6%)
  ✓ CustomCNN: failures guardados


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:1867: FutureWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  self._maybe_warn_non_full_backward_hook(args, result, grad_fn)


  ✓ ResNet-50: gradcam_good.png
  ✓ ResNet-50: gradcam_cracked.png
  Fallos encontrados: 5
    [1] Real:good → Pred:cracked (conf=90.2%)
    [2] Real:cracked → Pred:good (conf=88.9%)
    [3] Real:cracked → Pred:good (conf=60.2%)
    [4] Real:cracked → Pred:good (conf=59.1%)
    [5] Real:good → Pred:cracked (conf=54.3%)
  ✓ ResNet-50: failures guardados
  ✓ EfficientNet-B3: gradcam_good.png
  ✓ EfficientNet-B3: gradcam_cracked.png
  Fallos encontrados: 5
    [1] Real:cracked → Pred:good (conf=98.1%)
    [2] Real:good → Pred:cracked (conf=77.6%)
    [3] Real:good → Pred:cracked (conf=66.6%)
    [4] Real:good → Pred:cracked (conf=56.8%)
    [5] Real:good → Pred:cracked (conf=54.8%)
  ✓ EfficientNet-B3: failures guardados

✅ Grad-CAM completo
   Continúa con Celda 10 para descargar todo a tu Mac


## Celda 10 — Descargar resultados a tu Mac

### ¿Por qué es crítica esta celda?

Los servidores de Colab son temporales. Cuando la sesión
termina, **todos los archivos generados se eliminan para siempre**.

Esta celda comprime y descarga a tu Mac:

- **resultados_graficas.zip** → todas las figuras para el informe
- **modelos_entrenados.zip** → los 3 checkpoints entrenados

### Archivos que se descargan

**resultados_graficas.zip contiene:**
- results/comparativo/roc_comparativo.png
- results/comparativo/matrices_confusion.png
- results/comparativo/metricas_comparativo.png
- results/customcnn/gradcam_good.png
- results/customcnn/gradcam_cracked.png
- results/customcnn/failures/ (5 imágenes)
- results/resnet50/gradcam_good.png
- results/resnet50/gradcam_cracked.png
- results/resnet50/failures/ (5 imágenes)
- results/efficientnetb3/gradcam_good.png
- results/efficientnetb3/gradcam_cracked.png
- results/efficientnetb3/failures/ (5 imágenes)

**modelos_entrenados.zip contiene:**
- outputs/custom_cnn_aggressive_focal/best_model.pt
- outputs/resnet50_finetune_focal/best_model.pt
- outputs/efficientnet_b3_finetune_focal/best_model.pt
- outputs/*/test_metrics.json
- outputs/*/training_history.json

### ⚠️ Importante

Los archivos aparecerán en tu carpeta **Descargas** del Mac.
Guárdalos en tu carpeta del proyecto inmediatamente.
No cierres el navegador hasta que ambas descargas terminen.

In [20]:
import shutil, os
from google.colab import files

print("="*60)
print("📦 DESCARGANDO RESULTADOS A TU MAC")
print("="*60)

# ── Verificar que existen los archivos antes de comprimir ──────
print("\n📋 Verificando archivos generados...")

carpetas = [
    'results/comparativo',
    'results/customcnn',
    'results/resnet50',
    'results/efficientnetb3',
    'outputs/custom_cnn_aggressive_focal',
    'outputs/resnet50_finetune_focal',
    'outputs/efficientnet_b3_finetune_focal',
]

todos_ok = True
for carpeta in carpetas:
    existe = os.path.exists(carpeta)
    estado = "✓" if existe else "✗ NO ENCONTRADO"
    print(f"  {estado} {carpeta}")
    if not existe:
        todos_ok = False

if not todos_ok:
    print("\n⚠️ Algunas carpetas no existen.")
    print("   Verifica que las celdas anteriores se ejecutaron correctamente.")
else:
    print("\n✓ Todos los archivos verificados")

# ── Comprimir resultados ───────────────────────────────────────
print("\n🗜️  Comprimiendo resultados...")
shutil.make_archive(
    '/content/resultados_graficas',
    'zip',
    '/content/tire_classification/results'
)
size_r = os.path.getsize('/content/resultados_graficas.zip') / 1e6
print(f"  ✓ resultados_graficas.zip ({size_r:.1f} MB)")

print("\n🗜️  Comprimiendo modelos entrenados...")
shutil.make_archive(
    '/content/modelos_entrenados',
    'zip',
    '/content/tire_classification/outputs'
)
size_m = os.path.getsize('/content/modelos_entrenados.zip') / 1e6
print(f"  ✓ modelos_entrenados.zip ({size_m:.1f} MB)")

# ── Resumen de contenido ───────────────────────────────────────
print("\n📊 Resumen de archivos generados:")
total_results = 0
for root, dirs, fs in os.walk('results'):
    for f in fs:
        fp   = os.path.join(root, f)
        size = os.path.getsize(fp) / 1e3
        print(f"  {fp} ({size:.0f} KB)")
        total_results += 1

print(f"\n  Total archivos en results/: {total_results}")

# ── Descargar a Mac ────────────────────────────────────────────
print("\n⬇️  Iniciando descarga a tu Mac...")
print("   Los archivos aparecerán en tu carpeta Descargas")
print("   No cierres el navegador hasta que terminen\n")

files.download('/content/resultados_graficas.zip')
files.download('/content/modelos_entrenados.zip')

print("\n" + "="*60)
print("✅ PIPELINE COMPLETO")
print("="*60)
print("""
Resumen del proyecto:

  Dataset  : 1,028 imágenes (good: 491 | cracked: 537)
  Desbalance: 1.09x (prácticamente balanceado)

  Modelo          Params   F1      AUC     FN
  ─────────────────────────────────────────────
  CustomCNN        1.8M   0.6262  0.6891   34
  ResNet-50       24.0M   0.9681  0.9894    3
  EfficientNet-B3 11.1M   0.9681  0.9878    1 ← ganador seguridad

  Mejor modelo para producción: EfficientNet-B3
  - Igual F1 que ResNet-50 con la mitad de parámetros
  - Solo 1 falso negativo (llanta dañada no detectada)

Próximos pasos:
  1. Descomprimir los ZIPs en tu carpeta del proyecto
  2. Usar las figuras en el informe
  3. Hacer git push con el notebook actualizado
  4. Compilar el PDF del informe
""")

📦 DESCARGANDO RESULTADOS A TU MAC

📋 Verificando archivos generados...
  ✓ results/comparativo
  ✓ results/customcnn
  ✓ results/resnet50
  ✓ results/efficientnetb3
  ✓ outputs/custom_cnn_aggressive_focal
  ✓ outputs/resnet50_finetune_focal
  ✓ outputs/efficientnet_b3_finetune_focal

✓ Todos los archivos verificados

🗜️  Comprimiendo resultados...
  ✓ resultados_graficas.zip (21.4 MB)

🗜️  Comprimiendo modelos entrenados...
  ✓ modelos_entrenados.zip (138.1 MB)

📊 Resumen de archivos generados:
  results/comparativo/metricas_comparativo.png (80 KB)
  results/comparativo/roc_comparativo.png (135 KB)
  results/comparativo/matrices_confusion.png (111 KB)
  results/customcnn/gradcam_cracked.png (1432 KB)
  results/customcnn/gradcam_good.png (2137 KB)
  results/customcnn/failures/failure_03_training_data_NewNormal116.png (793 KB)
  results/customcnn/failures/failure_05_training_data_Untitled-381.png (367 KB)
  results/customcnn/failures/failure_04_testing_data_IMG_4424.png (557 KB)
  result

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ PIPELINE COMPLETO

Resumen del proyecto:

  Dataset  : 1,028 imágenes (good: 491 | cracked: 537)
  Desbalance: 1.09x (prácticamente balanceado)

  Modelo          Params   F1      AUC     FN
  ─────────────────────────────────────────────
  CustomCNN        1.8M   0.6262  0.6891   34
  ResNet-50       24.0M   0.9681  0.9894    3
  EfficientNet-B3 11.1M   0.9681  0.9878    1 ← ganador seguridad

  Mejor modelo para producción: EfficientNet-B3
  - Igual F1 que ResNet-50 con la mitad de parámetros
  - Solo 1 falso negativo (llanta dañada no detectada)

Próximos pasos:
  1. Descomprimir los ZIPs en tu carpeta del proyecto
  2. Usar las figuras en el informe
  3. Hacer git push con el notebook actualizado
  4. Compilar el PDF del informe



In [21]:
import os
from google.colab import files

# Verificar que los ZIPs existen
zip1 = '/content/resultados_graficas.zip'
zip2 = '/content/modelos_entrenados.zip'

print("Verificando ZIPs:")
print(f"  resultados_graficas.zip : {'✓ existe' if os.path.exists(zip1) else '✗ NO existe'}")
print(f"  modelos_entrenados.zip  : {'✓ existe' if os.path.exists(zip2) else '✗ NO existe'}")

if os.path.exists(zip1):
    size = os.path.getsize(zip1) / 1e6
    print(f"  Tamaño resultados: {size:.1f} MB")

if os.path.exists(zip2):
    size = os.path.getsize(zip2) / 1e6
    print(f"  Tamaño modelos: {size:.1f} MB")

Verificando ZIPs:
  resultados_graficas.zip : ✓ existe
  modelos_entrenados.zip  : ✓ existe
  Tamaño resultados: 21.4 MB
  Tamaño modelos: 138.1 MB


In [29]:
from google.colab import drive
import shutil

# Montar Google Drive
drive.mount('/content/drive')

# Crear carpeta en Drive
import os
os.makedirs('/content/drive/MyDrive/tire_classification_resultados', 
            exist_ok=True)

# Copiar los ZIPs a Drive
print("Copiando resultados (21 MB)...")
shutil.copy(
    '/content/resultados_graficas.zip',
    '/content/drive/MyDrive/tire_classification_resultados/resultados_graficas.zip'
)
print("✓ resultados_graficas.zip copiado")

print("Copiando modelos (132 MB)...")
shutil.copy(
    '/content/modelos_entrenados.zip',
    '/content/drive/MyDrive/tire_classification_resultados/modelos_entrenados.zip'
)
print("✓ modelos_entrenados.zip copiado")

print("\n✅ Listo — ve a drive.google.com y descarga los archivos")

Mounted at /content/drive
Copiando resultados (21 MB)...
✓ resultados_graficas.zip copiado
Copiando modelos (132 MB)...
✓ modelos_entrenados.zip copiado

✅ Listo — ve a drive.google.com y descarga los archivos
